# ChargeSquare Analytics — A1–A6

`energy_kwh` is the **cumulative** per-session meter register — the classic **energy trap**.
Every energy metric below uses per-session deltas, **never** `SUM(energy_kwh)`, which would
over-count by ~(readings per session).

This notebook runs the six analytical queries in `analytics/queries/` against ClickHouse
(over its HTTP interface, port 8123) and writes each result to `analytics/output/*.csv`.


In [1]:
import os
from pathlib import Path
import clickhouse_connect
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Resolve the pipeline root robustly: honour $PIPELINE_ROOT if set, otherwise walk
# upward from the current directory until we find the one that holds BOTH
# docker-compose.yml and analytics/queries. Works from the repo root, from analytics/,
# or from any nested subdirectory.
def find_root():
    env = os.environ.get("PIPELINE_ROOT")
    if env:
        return Path(env).resolve()
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "docker-compose.yml").exists() and (d / "analytics/queries").is_dir():
            return d
    raise RuntimeError(
        "could not locate pipeline root (need docker-compose.yml + analytics/queries); "
        "set PIPELINE_ROOT to override"
    )

ROOT = find_root()
QUERIES = ROOT / "analytics/queries"
OUTPUT = ROOT / "analytics/output"
OUTPUT.mkdir(parents=True, exist_ok=True)

client = clickhouse_connect.get_client(
    host=os.environ.get("CLICKHOUSE_HOST", "localhost"),
    port=int(os.environ.get("CLICKHOUSE_PORT", "8123")),
    username="chargesquare", password="chargesquare", database="ev",
)

LABELS = ["A1", "A2", "A3", "A4", "A5", "A6"]

def run(label):
    # clickhouse-connect wants a single statement, so strip the trailing ';'.
    sql = sorted(QUERIES.glob(f"{label}_*.sql"))[0].read_text(encoding="utf-8").strip().rstrip(";")
    df = client.query_df(sql)
    df.to_csv(OUTPUT / f"{label}.csv", index=False)
    print(f"{label}: {len(df)} rows -> analytics/output/{label}.csv")
    return df

def save_plot(name):
    try:
        plt.tight_layout(); plt.savefig(OUTPUT / f"{name}.png"); plt.show()
    except Exception as e:
        print("plot skipped:", e)
    finally:
        plt.close("all")


## A1 — Hourly energy consumption (per-session deltas, not cumulative sums)


In [2]:
a1 = run("A1")
try:
    a1.plot(x="hour", y="energy_kwh", figsize=(10,3), legend=False, title="A1 Hourly energy (kWh)")
    save_plot("A1")
except Exception as e:
    print("plot skipped:", e)
a1.head()


A1: 93 rows -> analytics/output/A1.csv


/var/folders/88/5j93ks7j4v3d_pf8hqr0_prr0000gn/T/ipykernel_73416/741225504.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT / f"{name}.png"); plt.show()


,hour,energy_kwh
0,2026-07-01 21:00:00,4708254.218
1,2026-07-01 22:00:00,5225692.694
2,2026-07-01 23:00:00,5405420.295
3,2026-07-02 00:00:00,5345772.636
4,2026-07-02 01:00:00,5320995.726


## A2 — Station uptime / downtime, worst stations per operator


In [3]:
a2 = run("A2")
try:
    top = a2.head(12).iloc[::-1]
    top.plot(kind="barh", x="station_id", y="downtime_s", figsize=(10,4), legend=False, title="A2 Downtime seconds (top stations)")
    save_plot("A2")
except Exception as e:
    print("plot skipped:", e)
a2.head(12)


A2: 20 rows -> analytics/output/A2.csv


/var/folders/88/5j93ks7j4v3d_pf8hqr0_prr0000gn/T/ipykernel_73416/741225504.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT / f"{name}.png"); plt.show()


,operator_id,station_id,total_s,downtime_s,uptime_ratio
0,VoltRun,TR-IST-16012,1322620,65961,0.9501
1,VoltRun,TR-IST-2235,1323152,56987,0.9569
2,Esarj,TR-IST-14558,1322620,56844,0.9570
3,ZES,TR-IZM-5350,1322620,56374,0.9574
4,ZES,TR-IST-1894,1323664,56123,0.9576
5,VoltRun,TR-BUR-2901,1322620,56095,0.9576
6,VoltRun,TR-IZM-1547,1323152,55606,0.9580
7,ZES,TR-IST-14890,1322620,55495,0.9580
8,ZES,TR-BUR-0352,1323664,54960,0.9585
9,ZES,TR-ANK-4245,1322620,54511,0.9588


## A3 — Average duration & energy by vehicle brand


In [4]:
a3 = run("A3")
try:
    a3.plot(kind="bar", x="brand", y=["avg_duration_min","avg_energy_kwh"], figsize=(10,3), title="A3 Avg duration (min) & energy (kWh) by brand")
    save_plot("A3")
except Exception as e:
    print("plot skipped:", e)
a3


A3: 9 rows -> analytics/output/A3.csv


/var/folders/88/5j93ks7j4v3d_pf8hqr0_prr0000gn/T/ipykernel_73416/741225504.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT / f"{name}.png"); plt.show()


,brand,sessions,avg_duration_min,avg_energy_kwh
0,Tesla,3843761,41.2,33.06
1,Volvo,1371009,42.4,29.26
2,Volkswagen,822946,43.5,29.64
3,Hyundai,732507,41.7,33.24
4,Kia,548994,41.7,33.24
5,Togg,548274,43.2,33.21
6,BMW,457575,42.2,32.35
7,Renault,457047,42.3,26.24
8,Mercedes-Benz,366127,44.1,25.33


## A4 — Revenue and peak-hour contribution, per operator


In [5]:
a4 = run("A4")
try:
    a4.plot(kind="bar", x="operator_id", y=["revenue_eur","peak_revenue_eur"], figsize=(10,3), title="A4 Revenue (EUR): total vs peak-hour")
    save_plot("A4")
except Exception as e:
    print("plot skipped:", e)
a4


A4: 140 rows -> analytics/output/A4.csv


/var/folders/88/5j93ks7j4v3d_pf8hqr0_prr0000gn/T/ipykernel_73416/741225504.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT / f"{name}.png"); plt.show()


,operator_id,city,tariff_id,revenue_eur,peak_revenue_eur,peak_pct,sessions
0,ZES,Istanbul,standard-v1,4460411.54,309912.06,6.9,362062
1,VoltRun,Istanbul,standard-v1,4441597.61,305352.67,6.9,363130
2,ChargeSquare,Istanbul,standard-v1,4414854.53,302352.10,6.8,361130
3,Trugo,Istanbul,standard-v1,4406373.22,303459.14,6.9,358433
4,Esarj,Istanbul,standard-v1,4394973.72,300255.61,6.8,356518
...,...,...,...,...,...,...,...
135,ZES,Konya,peak-rate-v2,113193.46,96810.50,85.5,5725
136,Trugo,Konya,peak-rate-v2,112764.91,96620.60,85.7,5691
137,ChargeSquare,Konya,peak-rate-v2,108283.22,92416.86,85.3,5686
138,Esarj,Konya,peak-rate-v2,107967.35,91657.24,84.9,5507


## A5 — Geographic distribution of FAULT events (deduped by event_id)


In [6]:
a5 = run("A5")
try:
    a5.plot(kind="bar", x="city", y="fault_count", figsize=(10,3), legend=False, title="A5 Fault count by city")
    save_plot("A5")
except Exception as e:
    print("plot skipped:", e)
a5


A5: 7 rows -> analytics/output/A5.csv


/var/folders/88/5j93ks7j4v3d_pf8hqr0_prr0000gn/T/ipykernel_73416/741225504.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT / f"{name}.png"); plt.show()


,city,fault_count,stations_affected,faults_per_station,lat,lon
0,Istanbul,203501,16995,11.97,41.0084,28.9786
1,Ankara,79783,6718,11.88,39.9337,32.8598
2,Izmir,66473,5570,11.93,38.4238,27.1433
3,Bursa,41452,3376,12.28,40.1886,29.0611
4,Antalya,39686,3363,11.80,36.8967,30.7134
5,Adana,26396,2199,12.00,37.0003,35.3211
6,Konya,19689,1697,11.60,37.8740,32.4914


## A6 — Anomaly detection: sessions > 2 sigma above the fleet mean power


In [7]:
a6 = run("A6")
try:
    a6.head(20).plot(kind="bar", x="session_id", y="z_score", figsize=(10,3), legend=False, title="A6 Anomalous sessions (z-score)")
    save_plot("A6")
except Exception as e:
    print("plot skipped:", e)
a6.head(20)


A6: 100 rows -> analytics/output/A6.csv


/var/folders/88/5j93ks7j4v3d_pf8hqr0_prr0000gn/T/ipykernel_73416/741225504.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT / f"{name}.png"); plt.show()


,session_id,station_id,brand,avg_power_kw,z_score
0,sess-d43ea836-3124-48cb,TR-IZM-0434,Tesla,250.0,4.75
1,sess-64d7c4b7-f07b-4b2c,TR-BUR-0292,Tesla,250.0,4.75
2,sess-1d364548-a086-48f4,TR-IZM-0215,Tesla,250.0,4.75
3,sess-0a4f065f-c5b2-4e03,TR-IST-1651,Tesla,250.0,4.75
4,sess-7a272bfb-9135-4796,TR-ANK-0619,Tesla,250.0,4.75
5,sess-36dda610-ee1c-41a3,TR-IZM-0397,Tesla,250.0,4.75
6,sess-13d88347-f552-4b0a,TR-IST-6502,Tesla,250.0,4.75
7,sess-3f1a0468-14b4-4b74,TR-IST-1971,Tesla,250.0,4.75
8,sess-17b9b48f-e356-4dec,TR-IST-0206,Tesla,250.0,4.75
9,sess-5f3e922b-0b94-4c7a,TR-ANK-0144,Tesla,250.0,4.75


## Summary

All six results are in `analytics/output/*.csv`. The energy figures (A1, A3) use
per-session **deltas** of the cumulative `energy_kwh` register — summing the raw column
would over-count by roughly the number of meter readings per session.
